# TrainLM T1.4: single-host TorchTPU distributed probe

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dhiraj309/TrainLM/blob/docs/01-t1-v5e1-probe/notebooks/TorchTPU_T1_v5e8_Distributed_Probe.ipynb)

Open this notebook in Colab to run or copy its code cells directly, including on mobile. This distributed probe requires a suitable v5e-8 runtime.

Use a v5e-8 TPU host for the optional collective. This is a bounded feasibility check, not distributed training or T4 certification. Run the single-device T1 notebook first. The run stays disabled until you choose to test it. Record the actual VM/accelerator identity and package versions with the result.


In [ ]:
from pathlib import Path
import subprocess
import sys
import tempfile

PROBE_BRANCH = "docs/01-t1-v5e1-probe"
PROBE = Path("examples/torchtpu_t1_distributed_probe.py")
repo_candidates = [Path.cwd(), *Path.cwd().parents, Path("/kaggle/working/TrainLM"), Path("/content/TrainLM")]
repo_root = next((p.resolve() for p in repo_candidates if (p / PROBE).is_file()), None)
if repo_root is None:
    clone_path = Path(tempfile.mkdtemp(prefix="trainlm-t1-distributed-")) / "TrainLM"
    subprocess.run(["git", "clone", "--depth", "1", "--branch", PROBE_BRANCH,
                    "https://github.com/Dhiraj309/TrainLM.git", str(clone_path)], check=True)
    repo_root = clone_path.resolve()
print(f"repository={repo_root}")
print(f"probe={repo_root / PROBE}")


## Read-only launch-environment inspection

An `incomplete` result is normal before TorchTPU prepares its launch environment. This cell does not import torch or initialize a TPU. Coordinator addresses are never printed.


In [ ]:
subprocess.run([sys.executable, str(repo_root / PROBE), "--inspect"],
               cwd=repo_root, check=True)


## Optional v5e-8 collective

Set `RUN_DISTRIBUTED = True` only on an isolated single-host v5e-8 runtime after the single-device smoke passes. TrainLM calls TorchTPU's launch helper in the parent process, starts eight workers, verifies rank/world consistency and one sum all-reduce, and tears the worker group down. It times out after 300 seconds. A failure is evidence to investigate; do not claim distributed support from a skipped run.


In [ ]:
RUN_DISTRIBUTED = False
if RUN_DISTRIBUTED:
    subprocess.run([sys.executable, str(repo_root / PROBE), "--run",
                    "--expected-world-size", "8", "--timeout", "300"],
                   cwd=repo_root, check=True)
else:
    print("distributed probe skipped; set RUN_DISTRIBUTED = True on a single-host v5e-8 runtime")
